In [ ]:
import cv2
import numpy as np
from time import sleep
import os  # Ensure the os module is imported

# Parameters
LINE_POSITION = 550  # Position of the line to detect crossing vehicles
OFFSET = 6           # Allowable error in line position for vehicle counting
DELAY = 60           # Frame rate delay in milliseconds

# Absolute paths to YOLO files
weights_path = r"C:\Users\varsh\Downloads\yolov4.weights"
config_path = r"C:\Users\varsh\Downloads\yolov4.cfg"

# Ensure weights and config files exist
if not os.path.exists(weights_path) or not os.path.exists(config_path):
    print("Error: YOLOv4 weights and/or configuration files are missing.")
    print(f"Expected at:\n - Weights: {weights_path}\n - Config: {config_path}")
    exit()

# Load YOLOv4 for vehicle detection
try:
    net = cv2.dnn.readNet(weights_path, config_path)
    layer_names = net.getLayerNames()
    output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]
except Exception as e:
    print(f"Error loading YOLOv4 files: {e}")
    exit()

# Full list of COCO classes for YOLOv4 (80 classes)
coco_classes = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck", "boat", "traffic light",
                "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
                "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
                "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
                "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
                "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa",
                "pottedplant", "bed", "diningtable", "toilet", "TV monitor", "laptop", "mouse", "remote", "keyboard",
                "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
                "teddy bear", "hair drier", "toothbrush"]

# Vehicle-related classes to be counted
vehicle_classes = ["car", "motorbike", "bus", "truck"]

# Initialize counters and detection list
detected_vehicle_count = 0
tracked_vehicle_ids = set()  # To ensure we don't recount the same vehicle

def get_center(x, y, w, h):
    """Calculate the center point of a rectangle (vehicle)."""
    cx = x + w // 2
    cy = y + h // 2
    return cx, cy

# Load video
cap = cv2.VideoCapture('data.mp4')  # Replace 'video.mp4' with your local video file path

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Adding a delay based on specified FPS
    sleep(1 / DELAY)

    # YOLO object detection
    height, width, channels = frame.shape
    blob = cv2.dnn.blobFromImage(frame, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
    net.setInput(blob)
    outs = net.forward(output_layers)

    boxes = []
    confidences = []
    class_ids = []

    # Process detections from YOLO
    for out in outs:
        for detection in out:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]
            if confidence > 0.5 and coco_classes[class_id] in vehicle_classes:
                # Object detected
                center_x = int(detection[0] * width)
                center_y = int(detection[1] * height)
                w = int(detection[2] * width)
                h = int(detection[3] * height)

                # Rectangle coordinates
                x = int(center_x - w / 2)
                y = int(center_y - h / 2)

                boxes.append([x, y, w, h])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    # Non-Maximum Suppression to remove overlapping boxes
    indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)
    for i in range(len(boxes)):
        if i in indexes:
            x, y, w, h = boxes[i]
            label = coco_classes[class_ids[i]]
            confidence = confidences[i]

            # Draw bounding box
            color = (0, 255, 0)
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            center = get_center(x, y, w, h)
            cv2.circle(frame, center, 4, (0, 0, 255), -1)

            # Check for crossing line
            if LINE_POSITION - OFFSET < center[1] < LINE_POSITION + OFFSET:
                # Assign a unique ID based on center coordinates for tracking
                vehicle_id = (center[0], center[1])
                if vehicle_id not in tracked_vehicle_ids:
                    detected_vehicle_count += 1
                    tracked_vehicle_ids.add(vehicle_id)  # Track this vehicle
                    cv2.line(frame, (25, LINE_POSITION), (1200, LINE_POSITION), (0, 127, 255), 3)
                    print("Detected Vehicle Count: " + str(detected_vehicle_count))

    # Display the frame
    cv2.imshow("Vehicle Detection", frame)

    # Exit on ESC key
    if cv2.waitKey(1) & 0xFF == 27:
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()


Detected Vehicle Count: 1
Detected Vehicle Count: 2
Detected Vehicle Count: 3
Detected Vehicle Count: 4
Detected Vehicle Count: 5
Detected Vehicle Count: 6
Detected Vehicle Count: 7
Detected Vehicle Count: 8
Detected Vehicle Count: 9
Detected Vehicle Count: 10
